# <span style="color: #1F1DB5;">SPRINT 13 – Proyecto del Módulo 2

# <span style="color: #1F1DB5;">Notebook 1 – Simulacro de Proyecto Integrado de Machine Learning — Versión estudiante

## Cómo usar esta versión estudiante

Este notebook está preparado para que tomes apuntes, completes ejercicios y documentes tus decisiones durante la clase.

**Modo de trabajo recomendado:**

1. Lee primero la consigna de cada bloque o ejercicio.
2. Completa los espacios de respuesta antes de comparar con la explicación del instructor/a.
3. Ejecuta las celdas de setup cuando existan.
4. Escribe interpretación, dudas y decisiones en los espacios de apuntes.
5. Al final, revisa si tus respuestas conectan datos, método e implicación de negocio.

> Las salidas de ejecución fueron limpiadas para que puedas correr el notebook desde cero.


## Notas de clase principales

- Integrar conocimientos de Sprints 9-12 en un proyecto completo de ML.
- Practicar el flujo profesional: EDA → Limpieza → Feature Engineering → Pipeline → Evaluación → Interpretación.
- Desarrollar habilidades de comunicación de resultados orientada a negocio.
- Aplicar una rúbrica de evaluación profesional para autoevaluarse.
- Trabajar en equipo simulando un entorno real de ciencia de datos.

### Mis notas iniciales

- 
- 
- 


## <span style="color: #2826DE;">Objetivos de Aprendizaje

- Integrar conocimientos de Sprints 9-12 en un **proyecto completo de ML**.
- Practicar el **flujo profesional**: EDA → Limpieza → Feature Engineering → Pipeline → Evaluación → Interpretación.
- Desarrollar habilidades de **comunicación de resultados** orientada a negocio.
- Aplicar una **rúbrica de evaluación** profesional para autoevaluarse.
- Trabajar en equipo simulando un entorno real de ciencia de datos.

### <span style="color: #2772F2;">Agenda (2 horas)

| Tema | Duración |
|---|---|
Introducción + contexto | 10 min |
Presentación del dataset y pregunta de negocio | 10 min |
Simulación de proyecto en equipos | 60 min |
Presentaciones de equipos | 20 min |
Rúbrica y retroalimentación | 10 min |
Cierre | 10 min |

## <span style="color: #2826DE;">❓ Pregunta Guía

### Si te dieran un dataset nuevo y una pregunta de negocio, ¿podrías entregar un modelo funcional en 60 minutos?

En este notebook vamos a poner a prueba todo lo que hemos aprendido en los Sprints 9 al 12. La idea es simular un proyecto real donde un cliente te pide resolver un problema de clasificación. Tendrás que tomar decisiones rápidas sobre limpieza, features y modelos.

### Mi respuesta inicial

- 


## <span style="color: #2826DE;">Recap: Lo que sabemos de Sprints 9-12 (10 mins)

Antes de lanzarnos al proyecto, recordemos las herramientas que tenemos disponibles:

**Sprint 9 – Machine Learning Intro:**
- Train/test split, overfitting vs underfitting.
- Árbol de decisión y Random Forest como primeros modelos.

**Sprint 10 – ML Supervisado:**
- Regresión logística, métricas de clasificación (precision, recall, F1).
- Validación cruzada para evaluación robusta.

**Sprint 11 – Feature Engineering:**
- One-Hot Encoding, Ordinal Encoding.
- Creación de features derivadas (ratios, interacciones, bins).

**Sprint 12 – Pipelines y Modelos:**
- `Pipeline` y `ColumnTransformer` para flujos reproducibles.
- `GridSearchCV` para optimización de hiperparámetros.

Hoy unimos **todo** esto en un solo flujo.

## <span style="color: #2826DE;">El Problema de Negocio (10 mins)

**Contexto:** Una empresa de telecomunicaciones quiere predecir qué clientes cancelarán su servicio (churn) en los próximos 30 días. Si logran identificarlos a tiempo, pueden ofrecer promociones de retención.

**Tu misión como equipo:**
1. Explorar y limpiar el dataset.
2. Crear features relevantes.
3. Construir un pipeline de clasificación.
4. Evaluar con métricas apropiadas (el negocio quiere minimizar falsos negativos: no perder clientes sin detectarlos).
5. Presentar resultados y recomendación.

**Métrica principal:** Recall (porque es peor no detectar un cliente que se va, que enviar una promoción innecesaria).

Vamos a generar un dataset realista de telecomunicaciones. Este código crea los datos que usarán los equipos. Ejecuten esta celda para obtener su DataFrame de trabajo.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

np.random.seed(42)
n = 1200

# Generamos datos de clientes de telecomunicaciones
data = {
    'antiguedad_meses': np.random.randint(1, 72, n),
    'contrato': np.random.choice(['Mensual', 'Anual', 'Bianual'], n, p=[0.5, 0.3, 0.2]),
    'cargo_mensual': np.round(np.random.uniform(20, 120, n), 2),
    'total_pagado': np.zeros(n),  # se calcula después
    'num_tickets_soporte': np.random.poisson(1.5, n),
    'tiene_fibra': np.random.choice([0, 1], n, p=[0.4, 0.6]),
    'metodo_pago': np.random.choice(['Tarjeta', 'Transferencia', 'Cheque electrónico', 'Cheque papel'], n),
    'edad': np.random.randint(18, 75, n),
    'dependientes': np.random.choice([0, 1, 2, 3], n, p=[0.4, 0.3, 0.2, 0.1]),
}

df = pd.DataFrame(data)
df['total_pagado'] = df['cargo_mensual'] * df['antiguedad_meses'] * np.random.uniform(0.85, 1.0, n)

# Crear churn con lógica realista (no lineal)
prob_churn = 0.1 + 0.3 * (df['contrato'] == 'Mensual').astype(int)
prob_churn += 0.15 * (df['num_tickets_soporte'] > 2).astype(int)
prob_churn -= 0.1 * (df['antiguedad_meses'] > 24).astype(int)
prob_churn += 0.1 * (df['cargo_mensual'] > 80).astype(int)
prob_churn = prob_churn.clip(0.05, 0.85)
df['churn'] = (np.random.random(n) < prob_churn).astype(int)

# Introducir algunos valores faltantes (realismo)
mask_cargo = np.random.random(n) < 0.03
df.loc[mask_cargo, 'cargo_mensual'] = np.nan
mask_edad = np.random.random(n) < 0.02
df.loc[mask_edad, 'edad'] = np.nan

print(f"Shape: {df.shape}")
print(f"Tasa de churn: {df['churn'].mean():.2%}")
print(f"Valores nulos:\n{df.isnull().sum()[df.isnull().sum() > 0]}")
df.head(10)

## <span style="color: #2826DE;">Simulación de Proyecto – Breakout Rooms (60 mins)

Ahora es su turno. Van a trabajar en equipos de 3-4 personas. Cada equipo debe completar el pipeline completo. A continuación tienen las instrucciones y celdas de trabajo.

### <span style="color: #aa0808;">En esta ocasión, quien compartirá resultados será: (Giremos la ruleta)

### Instrucciones para el equipo:

**Paso 1 – EDA (10 min):**
- Explorar distribuciones, correlaciones, balance de clases.

**Paso 2 – Limpieza (5 min):**
- Manejar nulos, verificar tipos de datos.

**Paso 3 – Feature Engineering (10 min):**
- Crear al menos 2 features nuevas (ratios, bins, interacciones).
- Encoding de variables categóricas.

**Paso 4 – Pipeline y Modelo (20 min):**
- Construir un `Pipeline` con `ColumnTransformer`.
- Probar al menos 2 modelos (ej. Logistic Regression + Random Forest).
- Usar `cross_val_score` con scoring='recall'.

**Paso 5 – Evaluación e Interpretación (15 min):**
- Reportar precision, recall, F1, matriz de confusión.
- ¿Qué features son más importantes?
- ¿Qué recomendación darían al negocio?

### Workspace – Paso 1: EDA

Explora el dataset. ¿Cómo se distribuye el churn? ¿Hay variables que parecen separar bien las clases?

### Espacio de trabajo del estudiante

**Respuesta, decisiones o interpretación:**

- 
- 


In [ ]:
# === PASO 1: EDA ===
# Tu código aquí
import matplotlib.pyplot as plt
import seaborn as sns

# Ejemplo: distribución de churn por tipo de contrato
# df.groupby('contrato')['churn'].mean().plot(kind='bar')
# plt.title('Tasa de Churn por Tipo de Contrato')
# plt.ylabel('Tasa de Churn')
# plt.show()


### Workspace – Paso 2: Limpieza

Maneja los valores nulos y prepara los datos para modelado.

### Espacio de trabajo del estudiante

**Respuesta, decisiones o interpretación:**

- 
- 


In [ ]:
# TODO estudiante: completa el código de acuerdo con la consigna anterior.
# Contexto: === PASO 2: LIMPIEZA ===


### Workspace – Paso 3: Feature Engineering

Crea features nuevas. Piensa: ¿qué relaciones entre variables podrían ser predictivas? Por ejemplo:
- `cargo_por_mes_antiguedad` = cargo_mensual / antiguedad_meses
- `tickets_per_month` = num_tickets_soporte / antiguedad_meses
- Bins de antigüedad (nuevo, medio, veterano)

### Espacio de trabajo del estudiante

**Respuesta, decisiones o interpretación:**

- 
- 


In [ ]:
# TODO estudiante: completa el código de acuerdo con la consigna anterior.
# Contexto: === PASO 3: FEATURE ENGINEERING ===


### Workspace – Paso 4: Pipeline y Modelo

Construye un pipeline con ColumnTransformer. Recuerda separar features numéricas y categóricas.

### Espacio de trabajo del estudiante

**Respuesta, decisiones o interpretación:**

- 
- 


In [ ]:
# === PASO 4: PIPELINE Y MODELO ===
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score
from sklearn.metrics import classification_report, confusion_matrix

# Tu código aquí



### Workspace – Paso 5: Evaluación

Reporta métricas finales y prepara tu interpretación para la presentación.

### Espacio de trabajo del estudiante

**Respuesta, decisiones o interpretación:**

- 
- 


In [ ]:
# TODO estudiante: completa el código de acuerdo con la consigna anterior.
# Contexto: === PASO 5: EVALUACIÓN ===


## <span style="color: #2826DE;">Presentaciones (20 mins)

Cada equipo tiene **4 minutos** para presentar:
1. ¿Qué features crearon y por qué?
2. ¿Qué modelo ganó y con qué recall?
3. ¿Cuál es su recomendación al negocio?

**Tips para la presentación:**
- Hablen de resultados, no de código.
- Usen números concretos: "Nuestro modelo detecta el 85% de los clientes que se van".
- Propongan una acción: "Recomendamos contactar a clientes con contrato mensual y más de 2 tickets".

## <span style="color: #2826DE;">Rúbrica de Evaluación (10 mins)

Esta es la rúbrica que usaremos para evaluar el proyecto integrado real. Úsenla como guía:

| Criterio | Excelente (3) | Bueno (2) | Necesita mejora (1) |
|---|---|---|---|
| **EDA** | Análisis completo con visualizaciones informativas | EDA básico, algunas visualizaciones | Solo `.describe()` y `.info()` |
| **Limpieza** | Manejo fundamentado de nulos y outliers | Manejo básico sin justificación | Datos sin limpiar |
| **Feature Engineering** | 3+ features relevantes con justificación | 1-2 features sin explicar por qué | Sin features nuevas |
| **Pipeline** | Pipeline completo con ColumnTransformer | Modelo sin pipeline formal | Código desordenado, no reproducible |
| **Evaluación** | Múltiples métricas, matriz de confusión, interpretación | Solo accuracy reportada | Sin evaluación formal |
| **Interpretación** | Recomendación clara con features importantes | Menciona importancia pero sin acción | Sin interpretación de negocio |
| **Código** | Limpio, comentado, reproducible | Funcional pero desordenado | Con errores o incompleto |

Preguntas:

- ¿Qué parte del pipeline les tomó más tiempo y por qué?

- ¿Cómo decidieron qué features crear? ¿Qué criterio usaron?

- Si tuvieran 30 minutos más, ¿qué mejorarían de su modelo?

### Mis respuestas de validación

- 
- 


## <span style="color: #2826DE;">Tips y buenas prácticas

- Siempre empieza por entender el **problema de negocio** antes de tocar los datos.
- La **métrica de evaluación** debe alinearse con el objetivo del negocio (no siempre es accuracy).
- Un pipeline bien construido te ahorra horas de debugging y garantiza **reproducibilidad**.
- En una entrevista técnica, lo que más importa es tu **proceso de pensamiento**, no solo el resultado.
- Documenta tus decisiones: ¿por qué eliminaste esa columna? ¿por qué ese threshold?

## <span style="color: #2826DE;">Cierre y Kahoot (5 mins)

Kahoot - Preguntas sugeridas

1️⃣ ¿Cuál es la métrica más apropiada cuando queremos minimizar falsos negativos?

- Precision
- Recall 
- Accuracy
- F1-Score


2️⃣ ¿Qué componente de sklearn permite combinar preprocesamiento numérico y categórico?

- Pipeline
- ColumnTransformer 
- GridSearchCV
- StandardScaler


3️⃣ ¿Por qué es importante hacer feature engineering antes de entrenar?

- Para que el modelo tenga más columnas siempre
- Para crear relaciones que el modelo no descubriría solo 
- Para eliminar todas las columnas originales
- Porque sklearn lo requiere obligatoriamente


4️⃣ ¿Qué significa un recall de 0.90 en un modelo de churn?

- El modelo acierta 90% de todas las predicciones
- De los clientes que sí se van, detecta al 90% 
- Solo 10% de predicciones positivas son correctas
- El modelo tiene 90% de accuracy


5️⃣ ¿Cuál es el riesgo de NO usar cross-validation?

- El modelo entrena más lento
- No puedes usar Random Forest
- Puedes tener una evaluación optimista o pesimista por un solo split 
- El modelo siempre hará overfitting


6️⃣ ¿Qué hace `Pipeline` de sklearn?

- Solo escala los datos
- Encadena pasos de preprocesamiento y modelo en un flujo reproducible 
- Divide automáticamente train y test
- Genera visualizaciones del modelo

## <span style="color: #2826DE;">Resumen: Flujo Profesional de un Proyecto de ML

En la industria, un proyecto de ciencia de datos sigue este flujo:

1. **Entender el negocio** → ¿Qué problema resuelvo? ¿Qué métrica importa?
2. **EDA** → Explorar datos, entender distribuciones, detectar anomalías.
3. **Limpieza** → Manejar nulos, outliers, formatos incorrectos.
4. **Feature Engineering** → Crear información nueva a partir de la existente.
5. **Modelado** → Pipeline reproducible con múltiples modelos.
6. **Evaluación** → Métricas alineadas al negocio, no solo accuracy.
7. **Interpretación** → ¿Qué aprendimos? ¿Qué recomendamos?
8. **Comunicación** → Presentar resultados de forma clara y accionable.

Este flujo es lo que evaluarán en su proyecto integrado real y lo que les preguntarán en entrevistas técnicas.